In [1]:
# 1. Install PySpark in Colab (The Big Data Engine)
!pip install pyspark

In [9]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, lower, regexp_replace, when, expr

print("Starting PySpark...")
spark = SparkSession.builder.appName("DataPreProcessing").getOrCreate()

# Load the raw data
print("Loading raw data...")
df = spark.read \
    .option("header", "true") \
    .option("inferSchema", "true") \
    .option("multiLine", "true") \
    .option("quote", "\"") \
    .option("escape", "\"") \
    .csv("grab_reviews_raw.csv")

# Use TRY_CAST to safely handle shifted text/booleans
df_clean = df.withColumn("score", expr("TRY_CAST(score AS INT)"))

# Drop any rows that are blank or became Null
df_clean = df_clean.dropna(subset=["content", "score"])

# PREPROCESSING: Clean the text
df_clean = df_clean.withColumn("content", lower(col("content")))
df_clean = df_clean.withColumn("content", regexp_replace(col("content"), "[^a-z\\s]", ""))

# Add the Sentiment Labels
print("Applying Sentiment Labels...")
df_clean = df_clean.withColumn("sentiment_category",
    when(col("score") >= 4, "Positive")
    .when(col("score") <= 2, "Negative")
    .otherwise("Neutral")
)

df_clean = df_clean.withColumn("sentiment_score_label",
    when(col("score") >= 4, 2.0)
    .when(col("score") <= 2, 0.0)
    .otherwise(1.0)
)

# REORDER COLUMNS
# Define the exact order for main columns
front_columns = [
    "review_id", "username", "app_version", "timestamp",
    "content", "score", "sentiment_category", "sentiment_score_label"
]

# Find any remaining columns and automatically put them at the end
remaining_columns = [c for c in df_clean.columns if c not in front_columns]

# Combine the lists and apply the new order
final_column_order = front_columns + remaining_columns
df_clean = df_clean.select(*final_column_order)

# Show the results to prove the order
print("\n--- Final Cleaned Data (Reordered) ---")
df_clean.show(5, truncate=30)

# Save the final file
csv_filename = "cleaned_data.csv"
df_clean.toPandas().to_csv(csv_filename, index=False)
print(f"\nSuccess! Saved as '{csv_filename}'.")

# Auto-download to computer
from google.colab import files
files.download(csv_filename)

Starting PySpark...
Loading raw data...
Applying Sentiment Labels...

--- Final Cleaned Data (Reordered) ---
+------------------------------+-------------+-----------+-------------------+------------------------------+-----+------------------+---------------------+---------------+----------+---------+------------------------------+-------------------+
|                     review_id|     username|app_version|          timestamp|                       content|score|sentiment_category|sentiment_score_label|thumbs_up_count|word_count|has_reply|                 reply_content|         replied_at|
+------------------------------+-------------+-----------+-------------------+------------------------------+-----+------------------+---------------------+---------------+----------+---------+------------------------------+-------------------+
|7a4bc5ad-aed5-4d7c-899c-762...|A Google user|    5.413.0|2026-06-12 12:24:51|i love using grab the app i...|    5|          Positive|                  2.0|

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>